> PostgreSQL 14+ practice. Run `sql/00_sample_data.sql` first, then `SET search_path TO sql_practice, public;` in this connection. Run cells in order in the same SQL session. See README.md for setup and expected results. These are SQL cells; the original Python notebook metadata did not make them executable as Python.


## ROW_NUMBER()

### Business requirement: Assign a sequential number to each customer's transactions, starting with their most recent transaction.

In [ ]:
-- Number each customer history newest first; transaction ID breaks same-date ties deterministically.
SELECT
    transaction_id,
    customer_id,
    amount,
    transaction_date,
    ROW_NUMBER() OVER (
        PARTITION BY
            customer_id
        ORDER BY
            transaction_date DESC,
            transaction_id DESC
    ) AS row_num
FROM payment_transactions
ORDER BY customer_id, row_num;

## RANK() and DENSE_RANK()

### Business requirement: Rank successful transactions by amount for each customer.

In [ ]:
-- ROW_NUMBER is unique; RANK skips positions after tied amounts; DENSE_RANK has no gaps.
SELECT
    transaction_id,
    customer_id, amount,
    ROW_NUMBER() OVER (
        PARTITION BY
            customer_id
        ORDER BY amount DESC, transaction_id ASC
    ) AS row_num,
    RANK() OVER (
        PARTITION BY
            customer_id
        ORDER BY amount DESC
    ) AS transaction_rank,
    DENSE_RANK() OVER (
        PARTITION BY
            customer_id
        ORDER BY amount DESC
    ) AS dense_rank
FROM payment_transactions
WHERE
    status = 'Success'
ORDER BY
    customer_id,
    amount DESC,
    transaction_id ASC;

## Retrieving the latest record per customer

### Business requirement: Build a customer activity snapshot containing only the most recent transaction for each customer.

In [ ]:
-- Rank each customer history and keep row 1 to return their latest payment.
WITH rank_transaction AS (
    SELECT transaction_id, customer_id, amount, status, transaction_date,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY transaction_date DESC, transaction_id DESC
        ) AS row_num
    FROM payment_transactions
)
SELECT transaction_id, customer_id, amount, transaction_date
FROM rank_transaction
WHERE row_num = 1
ORDER BY customer_id;

## Month-over-month growth



### Business requirement: Calculate monthly successful payment volume and percentage growth compared with the previous month.

In [ ]:
-- LAG compares adjacent months present in the data; missing calendar months are not filled.
WITH monthly_revenue AS (
    SELECT DATE_TRUNC('month', transaction_date) AS month, SUM(amount) AS revenue
    FROM payment_transactions
    WHERE status = 'Success'
    GROUP BY DATE_TRUNC('month', transaction_date)
), monthly_comparison AS (
    SELECT month, revenue, LAG(revenue) OVER (ORDER BY month) AS previous_revenue
    FROM monthly_revenue
)
SELECT month, revenue, previous_revenue,
    revenue - previous_revenue AS revenue_difference,
    ROUND(100.0 * (revenue - previous_revenue) / NULLIF(previous_revenue, 0), 2) AS growth_percentage
FROM monthly_comparison
ORDER BY month;